# Instagram Analytics: Advanced Analysis & Machine Learning

## Introduction
In this final notebook, we will move beyond historical analysis and build a predictive model. Our goal is to predict the `performance_bucket_label` (low, medium, high, viral) of a post *before* it goes live. 

To do this properly, we will only use features known prior to posting (e.g., `follower_count`, `media_type`, `post_hour`, `hashtags_count`) and exclude outcome metrics (like `likes` or `comments`) to prevent data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

## 1. Load Data and Select Features
We will load our cleaned dataset and select our predictor variables (X) and our target variable (y).

In [ ]:
# Load the processed data
df = pd.read_csv('../data/processed/cleaned_instagram_data.csv')

# Select predictive features (avoiding leakage from likes, shares, etc.)
features = [
    'account_type', 'follower_count', 'media_type', 'content_category', 
    'traffic_source', 'has_call_to_action', 'post_hour', 'day_of_week', 
    'caption_length', 'hashtags_count'
]

X = df[features].copy()
y = df['performance_bucket_label'].copy()

## 2. Data Preprocessing
Machine learning models require numerical input. We need to convert our categorical text columns (like `media_type` and `day_of_week`) into numbers using One-Hot Encoding.

In [ ]:
# Convert categorical variables into dummy/indicator variables
categorical_cols = ['account_type', 'media_type', 'content_category', 'traffic_source', 'day_of_week']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Display the new shape to see the expanded feature set
print("Original features shape:", X.shape)
print("Encoded features shape:", X_encoded.shape)

## 3. Train/Test Split
We will split our data so that the model trains on 80% of the dataset and tests its accuracy on the remaining 20%.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size: {X_test.shape[0]} rows")

## 4. Train the Random Forest Model
Random Forest is a robust algorithm that handles tabular data exceptionally well and provides insight into feature importance.

In [ ]:
# Initialize and train the classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

print("Model training complete!")

## 5. Model Evaluation
Let's check how accurately our model predicts the performance buckets on unseen data.

In [ ]:
y_pred = rf_model.predict(X_test)

# Print classification report for precision, recall, and f1-score
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Plot the confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=['low', 'medium', 'high', 'viral'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['low', 'medium', 'high', 'viral'], yticklabels=['low', 'medium', 'high', 'viral'])
plt.title('Confusion Matrix')
plt.ylabel('Actual Bucket')
plt.xlabel('Predicted Bucket')
plt.show()

## 6. Feature Importance
Which variables are the strongest predictors of a post going viral? Let's extract the feature importances from our Random Forest.

In [ ]:
importances = rf_model.feature_importances_
feature_names = X_encoded.columns

# Create a DataFrame for easy plotting
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(15) # Top 15 features

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Most Important Features for Predicting Post Performance')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()